# BÀI THỰC HÀNH CHƯƠNG 3: TRAINING DEEP NEURAL NETWORKS VỚI PYTORCH
**Môn học:** Deep Learning / Machine Learning - UTH
**Giảng viên:** PhD. Nguyễn Thị Khánh Tiên

---
### Mục tiêu thực hành (Slide 23):
1. **Thực nghiệm với nhiều pre-trained models**: ResNet-18, VGG-16, DenseNet-121.
2. **Phân biệt & triển khai**: Transfer Learning (Feature Extraction) và Fine-Tuning.
3. **Điều chỉnh siêu tham số (Hyperparameters)**: Learning rate, Batch size, Optimizers (Adam vs SGD), Regularization.
4. **Giám sát bằng TensorBoard**: Theo dõi Loss, Accuracy, Model Graph, Sample Images, Weight Histograms.
5. **Save & Load Model**: Lưu và phục hồi checkpoint mô hình.

## 1. Thiết lập môi trường & Kiểm tra thiết bị (GPU/CPU)

In [3]:
import os
import time
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
import torchvision.utils as vutils
from torch.utils.tensorboard import SummaryWriter

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch version: {torch.__version__}')
print(f'Thiết bị huấn luyện: {device}')

PyTorch version: 2.15.0+cpu
Thiết bị huấn luyện: cpu


## 2. Tải và Khám phá Kiến trúc Mô hình (ResNet, VGG, DenseNet)
Mỗi kiến trúc mạng có vị trí tầng classification layer khác nhau:
- `ResNet18`: `model.fc = nn.Linear(...)`
- `VGG16`: `model.classifier[6] = nn.Linear(...)`
- `DenseNet121`: `model.classifier = nn.Linear(...)`

In [4]:
# Khởi tạo cấu trúc các model
resnet = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
print('ResNet-18 Classifier:', resnet.fc)

vgg = models.vgg16(weights=models.VGG16_Weights.DEFAULT)
print('VGG-16 Classifier cuối:', vgg.classifier[6])

densenet = models.densenet121(weights=models.DenseNet121_Weights.DEFAULT)
print('DenseNet-121 Classifier:', densenet.classifier)

ResNet-18 Classifier: Linear(in_features=512, out_features=1000, bias=True)
Downloading: "https://download.pytorch.org/models/vgg16-397923af.pth" to C:\Users\tan/.cache\torch\hub\checkpoints\vgg16-397923af.pth


100.0%
5.3%

VGG-16 Classifier cuối: Linear(in_features=4096, out_features=1000, bias=True)
Downloading: "https://download.pytorch.org/models/densenet121-a639ec97.pth" to C:\Users\tan/.cache\torch\hub\checkpoints\densenet121-a639ec97.pth


100.0%


DenseNet-121 Classifier: Linear(in_features=1024, out_features=1000, bias=True)


## 3. Hàm tùy biến Model: Transfer Learning vs Fine-Tuning

In [5]:
def get_model(model_name='resnet18', num_classes=10, mode='transfer_learning'):
    if model_name == 'resnet18':
        model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        for p in model.parameters():
            p.requires_grad = False
        if mode == 'fine_tune':
            for p in model.layer4.parameters():
                p.requires_grad = True
        model.fc = nn.Linear(model.fc.in_features, num_classes)
        
    elif model_name == 'vgg16':
        model = models.vgg16(weights=models.VGG16_Weights.DEFAULT)
        for p in model.parameters():
            p.requires_grad = False
        if mode == 'fine_tune':
            for p in model.features[24:].parameters():
                p.requires_grad = True
        model.classifier[6] = nn.Linear(model.classifier[6].in_features, num_classes)
        
    elif model_name == 'densenet121':
        model = models.densenet121(weights=models.DenseNet121_Weights.DEFAULT)
        for p in model.parameters():
            p.requires_grad = False
        if mode == 'fine_tune':
            for p in model.features.denseblock4.parameters():
                p.requires_grad = True
        model.classifier = nn.Linear(model.classifier.in_features, num_classes)
        
    total_p = sum(p.numel() for p in model.parameters())
    trainable_p = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f'[{model_name.upper()} - {mode}] Total: {total_p:,} | Trainable: {trainable_p:,} ({trainable_p/total_p*100:.2f}%)')
    return model

# So sánh tham số giữa 2 chế độ
m1 = get_model('resnet18', mode='transfer_learning')
m2 = get_model('resnet18', mode='fine_tune')

[RESNET18 - transfer_learning] Total: 11,181,642 | Trainable: 5,130 (0.05%)
[RESNET18 - fine_tune] Total: 11,181,642 | Trainable: 8,398,858 (75.11%)


## 4. Chuẩn bị dữ liệu CIFAR-10 & DataLoaders

In [6]:
transform_train = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

transform_test = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

train_set = datasets.CIFAR10(root='./data', train=True, download=True, transform=transform_train)
test_set = datasets.CIFAR10(root='./data', train=False, download=True, transform=transform_test)

train_loader = DataLoader(train_set, batch_size=64, shuffle=True, num_workers=0)
test_loader = DataLoader(test_set, batch_size=64, shuffle=False, num_workers=0)
print(f'Train samples: {len(train_set)}, Test samples: {len(test_set)}')

Train samples: 50000, Test samples: 10000


## 5. Cấu hình Huấn luyện và TensorBoard Monitoring

In [7]:
model_name = 'resnet18'
mode = 'transfer_learning'
lr = 0.001
weight_decay = 1e-4

model = get_model(model_name=model_name, num_classes=10, mode=mode).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam([p for p in model.parameters() if p.requires_grad], lr=lr, weight_decay=weight_decay)

exp_name = f'{model_name}_{mode}_adam_lr{lr}'
writer = SummaryWriter(f'runs/{exp_name}')

# Lưu ảnh mẫu vào TensorBoard
sample_imgs, _ = next(iter(train_loader))
writer.add_image('Sample_Inputs', vutils.make_grid(sample_imgs[:16], normalize=True))
print(f'TensorBoard log lưu tại: runs/{exp_name}')

[RESNET18 - transfer_learning] Total: 11,181,642 | Trainable: 5,130 (0.05%)
TensorBoard log lưu tại: runs/resnet18_transfer_learning_adam_lr0.001


## 6. Training & Validation Loop
Huấn luyện mô hình và log độ chính xác, hàm mất mát lên TensorBoard.

In [8]:
epochs = 3

for epoch in range(1, epochs + 1):
    # --- Huấn luyện ---
    model.train()
    train_loss, train_correct, train_total = 0.0, 0, 0
    for i, (inputs, targets) in enumerate(train_loader):
        inputs, targets = inputs.to(device), targets.to(device)
        
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item() * inputs.size(0)
        _, predicted = outputs.max(1)
        train_total += targets.size(0)
        train_correct += predicted.eq(targets).sum().item()
        
        if (i + 1) % 150 == 0:
            print(f'Epoch [{epoch}/{epochs}] Batch [{i+1}/{len(train_loader)}] Loss: {loss.item():.4f}')
            
    avg_train_loss = train_loss / train_total
    train_acc = 100.0 * train_correct / train_total
    
    # --- Đánh giá (Validation) ---
    model.eval()
    val_loss, val_correct, val_total = 0.0, 0, 0
    with torch.no_grad():
        for inputs, targets in test_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            
            val_loss += loss.item() * inputs.size(0)
            _, predicted = outputs.max(1)
            val_total += targets.size(0)
            val_correct += predicted.eq(targets).sum().item()
            
    avg_val_loss = val_loss / val_total
    val_acc = 100.0 * val_correct / val_total
    
    # Ghi nhận lên TensorBoard
    writer.add_scalar('Loss/train', avg_train_loss, epoch)
    writer.add_scalar('Accuracy/train', train_acc, epoch)
    writer.add_scalar('Loss/val', avg_val_loss, epoch)
    writer.add_scalar('Accuracy/val', val_acc, epoch)
    
    print(f'>>> KẾT THÚC EPOCH {epoch}: Train Loss: {avg_train_loss:.4f}, Train Acc: {train_acc:.2f}% | Val Loss: {avg_val_loss:.4f}, Val Acc: {val_acc:.2f}%')

writer.close()
print('Huấn luyện hoàn tất!')

Epoch [1/3] Batch [150/782] Loss: 1.2986
Epoch [1/3] Batch [300/782] Loss: 1.1838
Epoch [1/3] Batch [450/782] Loss: 1.3081
Epoch [1/3] Batch [600/782] Loss: 0.8890
Epoch [1/3] Batch [750/782] Loss: 1.2610
>>> KẾT THÚC EPOCH 1: Train Loss: 1.2279, Train Acc: 58.28% | Val Loss: 1.0958, Val Acc: 63.61%
Epoch [2/3] Batch [150/782] Loss: 1.0312
Epoch [2/3] Batch [300/782] Loss: 0.8240
Epoch [2/3] Batch [450/782] Loss: 1.0903
Epoch [2/3] Batch [600/782] Loss: 1.0019
Epoch [2/3] Batch [750/782] Loss: 0.9190
>>> KẾT THÚC EPOCH 2: Train Loss: 1.0772, Train Acc: 63.29% | Val Loss: 1.0592, Val Acc: 64.58%
Epoch [3/3] Batch [150/782] Loss: 1.1747
Epoch [3/3] Batch [300/782] Loss: 1.1094
Epoch [3/3] Batch [450/782] Loss: 1.0417
Epoch [3/3] Batch [600/782] Loss: 0.8561
Epoch [3/3] Batch [750/782] Loss: 1.2287
>>> KẾT THÚC EPOCH 3: Train Loss: 1.0572, Train Acc: 64.03% | Val Loss: 1.0383, Val Acc: 65.56%
Huấn luyện hoàn tất!


## 7. Lưu và Tải Checkpoint Mô hình
Sử dụng `torch.save` và `model.load_state_dict(torch.load(...))`.

In [9]:
os.makedirs('checkpoints', exist_ok=True)
ckpt_file = 'checkpoints/best_model.pth'
torch.save(model.state_dict(), ckpt_file)
print(f'Đã lưu checkpoint: {ckpt_file}')

# Phục hồi mô hình
restored_model = get_model('resnet18', num_classes=10, mode='transfer_learning').to(device)
restored_model.load_state_dict(torch.load(ckpt_file))
restored_model.eval()
print('Tải lại mô hình thành công!')

Đã lưu checkpoint: checkpoints/best_model.pth
[RESNET18 - transfer_learning] Total: 11,181,642 | Trainable: 5,130 (0.05%)
Tải lại mô hình thành công!


## 8. Giám sát trên giao diện Web TensorBoard
Để mở giao diện trực quan TensorBoard:
```bash
tensorboard --logdir=runs
```
Sau đó mở trình duyệt web truy cập `http://localhost:6006`.